In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/xgb_shelf_life_model.json
/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/xgb_categorical_encoder.pkl
/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/xgb_feature_names.pkl
/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/freshness_labels.json
/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/best_food_category_model/config.json
/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/best_food_category_model/metadata.json
/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/best_food_category_model/model.weights.h5
/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/best_freshness_model/config.json
/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/best_freshness_model/metadata.json
/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/best_freshness_model/model.weights.h

In [2]:
# =====================================================================
# CELL 1: INSTALL REQUIRED LIBRARIES
# Run this cell first to set up the Kaggle environment.
# =====================================================================
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio sqlalchemy opencv-python-headless ultralytics xgboost scikit-learn pandas
print("✓ All dependencies installed successfully.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.4 MB/s eta 0:00:00
✓ All dependencies installed successfully.


In [3]:
# =====================================================================
# CELL 2: DATABASE SCHEMA & INITIALIZATION
# =====================================================================
import os
import hashlib
from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, ForeignKey, Text
from sqlalchemy.orm import declarative_base, sessionmaker, relationship

# Create the SQLite database file inside Kaggle's working directory
DATABASE_URL = "sqlite:////kaggle/working/food_freshness.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

# --- TABLES ---
class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True, index=True)
    name = Column(String(100), nullable=False)
    email = Column(String(100), unique=True, index=True, nullable=False)
    password_hash = Column(String(255), nullable=False)
    role = Column(String(50), default="user")
    created_at = Column(DateTime, default=datetime.utcnow)
    food_items = relationship("FoodItem", back_populates="user")

class FoodItem(Base):
    __tablename__ = "food_items"
    id = Column(Integer, primary_key=True, index=True)
    user_id = Column(Integer, ForeignKey("users.id"), nullable=False)
    name = Column(String(100), nullable=False)
    category = Column(String(50), nullable=False)
    purchase_date = Column(DateTime, default=datetime.utcnow)
    quantity = Column(String(50), default="1")
    storage_type = Column(String(50), default="Room")
    temperature = Column(Float, default=25.0)
    humidity = Column(Float, default=60.0)
    status = Column(String(50), default="Fresh")
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)
    user = relationship("User", back_populates="food_items")
    images = relationship("FoodImage", back_populates="food_item")

class FoodImage(Base):
    __tablename__ = "food_images"
    id = Column(Integer, primary_key=True, index=True)
    food_item_id = Column(Integer, ForeignKey("food_items.id"), nullable=False)
    image_url_path = Column(String(255), nullable=False)
    uploaded_at = Column(DateTime, default=datetime.utcnow)
    food_item = relationship("FoodItem", back_populates="images")
    prediction = relationship("AiPrediction", back_populates="food_image", uselist=False)

class AiPrediction(Base):
    __tablename__ = "ai_predictions"
    id = Column(Integer, primary_key=True, index=True)
    food_image_id = Column(Integer, ForeignKey("food_images.id"), nullable=False)
    predicted_class = Column(String(100), nullable=False)
    freshness_label = Column(String(50), nullable=False)
    freshness_score = Column(Float, nullable=False)
    remaining_shelf_life = Column(Float, nullable=False)
    spoilage_probability = Column(Float, nullable=False)
    model_version = Column(String(50), default="v2.0-full-stack")
    predicted_at = Column(DateTime, default=datetime.utcnow)
    food_image = relationship("FoodImage", back_populates="prediction")
# Add this below your User class in the database cell
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

# Build the database
Base.metadata.create_all(bind=engine)
print("✓ Database tables initialized.")

✓ Database tables initialized.


In [4]:
# =====================================================================
# CELL 3: Keras AI Models & OpenCV Highlighter Engine
# =====================================================================
import cv2
import numpy as np
import joblib
import pandas as pd
import base64
import json
import xgboost as xgb
import tensorflow as tf

# 1. EXACT KAGGLE PATHS FROM YOUR UPLOADED DATASET

CATEGORY_MODEL_PATH   = "/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/best_food_category_model"
FRESHNESS_MODEL_PATH  = "/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/best_freshness_model"
FRESHNESS_LABELS_PATH = "/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/freshness_labels.json"

XGB_MODEL_PATH = "/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/xgb_shelf_life_model.json"
ENCODER_PATH   = "/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/xgb_categorical_encoder.pkl"
FEATURES_PATH  = "/kaggle/input/datasets/ajinkyasupekar/food-freshness-backend-models/xgb_feature_names.pkl"

print("Loading AI Models into memory...")
try:
    # A. Load Keras Deep Learning Models
    category_model = tf.keras.models.load_model(CATEGORY_MODEL_PATH)
    freshness_model = tf.keras.models.load_model(FRESHNESS_MODEL_PATH)
    
    # B. Load Freshness Dictionary (e.g., {"0": "Fresh", "1": "Spoiled"})
    with open(FRESHNESS_LABELS_PATH, 'r') as f:
        freshness_labels = json.load(f)
        
    # C. Category Classes (Must match the order they were trained in)
    CATEGORY_CLASSES = ['Apple', 'Banana', 'Orange', 'Mango', 'Potato', 'Tomato', 'Carrot']

    # D. Load XGBoost Shelf-Life Models
    xgb_model = xgb.XGBRegressor()
    xgb_model.load_model(XGB_MODEL_PATH)
    xgb_encoder = joblib.load(ENCODER_PATH)
    xgb_features = joblib.load(FEATURES_PATH)
    
    print("✓ All Keras & XGBoost AI Models Loaded Successfully!")
    
except Exception as e:
    print(f"⚠️ Warning: Could not load some models. Error: {e}")
    xgb_model = None

# -------------------------------------------------------------
# 2. OPENCV DAMAGE HIGHLIGHTER
# -------------------------------------------------------------
def calculate_spoilage_and_highlight(image_bytes: bytes):
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    if img is None: return 0.0, [], None

    img_resized = cv2.resize(img, (640, 640))
    hsv = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)

    # Masks for fruit vs rot
    fruit_mask = cv2.inRange(hsv, np.array([0, 30, 30]), np.array([180, 255, 255]))
    total_pixels = cv2.countNonZero(fruit_mask)
    if total_pixels == 0: return 0.0, [], None

    decay_mask = cv2.inRange(hsv, np.array([5, 50, 20]), np.array([25, 255, 120]))
    kernel = np.ones((5, 5), np.uint8)
    decay_clean = cv2.morphologyEx(decay_mask, cv2.MORPH_CLOSE, kernel)
    decay_fruit = cv2.bitwise_and(decay_clean, decay_clean, mask=fruit_mask)

    spoilage_pct = round((cv2.countNonZero(decay_fruit) / total_pixels) * 100.0, 2)
    contours, _ = cv2.findContours(decay_fruit, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    damage_boxes = []
    highlighted_img = img_resized.copy() # Copy to draw on
    
    for cnt in contours:
        if cv2.contourArea(cnt) > 150:
            x, y, w, h = cv2.boundingRect(cnt)
            damage_boxes.append({"x": int(x), "y": int(y), "width": int(w), "height": int(h)})
            # Draw Yellow Outline and Red Box
            cv2.drawContours(highlighted_img, [cnt], -1, (0, 255, 255), 2)
            cv2.rectangle(highlighted_img, (x, y), (x+w, y+h), (0, 0, 255), 2)
            cv2.putText(highlighted_img, "Spoiled", (x, max(y-10, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    return min(100.0, max(0.0, spoilage_pct)), damage_boxes, highlighted_img

Loading AI Models into memory...


2026-08-19 04:39:28.441737: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


✓ All Keras & XGBoost AI Models Loaded Successfully!


In [5]:
# =====================================================================
# CELL 4: COMPLETE API ROUTE & HELPER FUNCTIONS (COLUMN MATCH FIX)
# =====================================================================
import os
import io
import cv2
import json
import base64
import traceback
import numpy as np
import pandas as pd
from PIL import Image
from fastapi import FastAPI, UploadFile, File, Form, Depends, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from sqlalchemy.orm import Session

# 1. Initialize FastAPI & Enable CORS
app = FastAPI(title="AI Food Freshness API", version="3.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# 2. OpenCV Damage Detection Helper Function
def detect_damage(image_path: str):
    img = cv2.imread(image_path)
    if img is None:
        return 0.0, [], None

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    lower_dark = np.array([0, 0, 0])
    upper_dark = np.array([180, 255, 80])
    mask = cv2.inRange(hsv, lower_dark, upper_dark)

    total_pixels = img.shape[0] * img.shape[1]
    damaged_pixels = cv2.countNonZero(mask)
    spoilage_pct = round((damaged_pixels / total_pixels) * 100, 2)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    damage_boxes = []
    highlighted_img = img.copy()

    for cnt in contours:
        if cv2.contourArea(cnt) > 150:
            x, y, w, h = cv2.boundingRect(cnt)
            damage_boxes.append({
                "x": int(x), "y": int(y), "width": int(w), "height": int(h)
            })
            cv2.rectangle(highlighted_img, (x, y), (x + w, y + h), (0, 0, 255), 2)
            cv2.putText(highlighted_img, "Spoiled", (x, max(y - 10, 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    _, buffer = cv2.imencode('.jpg', highlighted_img)
    base64_str = base64.b64encode(buffer).decode('utf-8')

    return min(100.0, max(0.0, spoilage_pct)), damage_boxes, base64_str

# 3. Storage Suggestions Generator Helper Function
def generate_storage_suggestions(food_name: str, freshness_status: str, predicted_life: float, spoilage_pct: float):
    food = str(food_name).lower()
    
    if "spoil" in freshness_status.lower() or spoilage_pct > 35.0:
        return {
            "recommended_action": "Discard or Compost Immediately",
            "ideal_storage_location": "Do not store with fresh produce",
            "handling_tips": [
                "Separate from other produce immediately to avoid ethylene gas spread.",
                "Dispose of safely in organic waste/compost."
            ]
        }
    
    if any(item in food for item in ["apple", "orange", "carrot", "cabbage"]):
        location = "Crisper Drawer (Refrigerator, 2-4°C)"
        action = "Safe to store long-term"
        tips = ["Keep in a perforated plastic bag to maintain moisture.", f"Expected shelf life is approximately {predicted_life} days."]
    elif any(item in food for item in ["banana", "mango", "potato", "tomato"]):
        location = "Cool, dry pantry or countertop (15-20°C)"
        action = "Consume within recommended days"
        tips = ["Do not refrigerate until fully ripe.", f"Consume within {predicted_life} days for best flavor."]
    else:
        location = "Standard Refrigeration or Cool Pantry"
        action = "Monitor daily"
        tips = ["Maintain moderate humidity levels.", f"Estimated shelf life remaining: {predicted_life} days."]
        
    return {"recommended_action": action, "ideal_storage_location": location, "handling_tips": tips}

# 4. Main Inference API Route
@app.post("/api/v1/food/analyze")
async def analyze_food(
    user_id: int = Form(1),
    file: UploadFile = File(...),
    temperature: float = Form(25.0),
    humidity: float = Form(60.0),
    storage_type: str = Form("Room"),
    db: Session = Depends(get_db)
):
    temp_path = None
    try:
        # A. Save uploaded file temporarily
        os.makedirs("/kaggle/working/uploads", exist_ok=True)
        temp_path = f"/kaggle/working/uploads/temp_{file.filename}"
        
        image_bytes = await file.read()
        with open(temp_path, "wb") as f:
            f.write(image_bytes)
            
        # B. Preprocess image for Deep Learning models
        img = Image.open(io.BytesIO(image_bytes)).convert('RGB').resize((224, 224))
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        # C. Keras Food Category Prediction
        category_pred = category_model.predict(img_array, verbose=0)
        food_idx = int(np.argmax(category_pred[0]))
        food_name = CATEGORY_CLASSES[food_idx] if food_idx < len(CATEGORY_CLASSES) else "Food Item"
        
        # D. Keras Freshness Prediction
        freshness_pred = freshness_model.predict(img_array, verbose=0)
        fresh_idx = str(int(np.argmax(freshness_pred[0])))
        confidence_score = float(np.max(freshness_pred[0]))
        
        freshness_label = freshness_labels.get(fresh_idx, "Unknown")
        if isinstance(freshness_label, dict) and "status" in freshness_label:
            freshness_status = freshness_label["status"]
        else:
            freshness_status = str(freshness_label)
            
        # E. OpenCV Surface Damage
        spoilage_pct, damage_boxes, highlighted_img = detect_damage(temp_path)
        spoilage_pct = float(spoilage_pct)
        
        # F. XGBoost Shelf-Life Prediction (FIXED COLUMN NAMES)
        if xgb_model is not None:
            # 1. EXACT column names for Categorical Encoder
            cat_data = pd.DataFrame({
                'Fruit_Vegetable': [food_name],
                'Storage_Condition': [str(storage_type)]
            })
            
            # 2. Both Uppercase and Lowercase numerical columns to be perfectly safe
            num_data = pd.DataFrame({
                'Temperature': [float(temperature)],
                'temperature': [float(temperature)],
                'Humidity': [float(humidity)],
                'humidity': [float(humidity)],
                'Freshness_Score': [float(confidence_score)],
                'freshness_score': [float(confidence_score)]
            })
            
            # 3. Transform categorical data
            encoded_cats = xgb_encoder.transform(cat_data)
            if hasattr(encoded_cats, "toarray"):
                encoded_cats = encoded_cats.toarray()
            encoded_df = pd.DataFrame(encoded_cats, columns=xgb_encoder.get_feature_names_out())
            
            # 4. Combine all features
            final_features = pd.concat([num_data, encoded_df], axis=1)
            
            # 5. Extract exactly what XGBoost wants, filling missing with 0
            for col in xgb_features:
                if col not in final_features.columns:
                    final_features[col] = 0.0
                    
            final_features = final_features[xgb_features]
            
            # Predict
            predicted_life = float(xgb_model.predict(final_features)[0])
            predicted_life = max(0.0, round(predicted_life, 1))
        else:
            predicted_life = 0.0

        # G. Generate Storage Guidance
        suggestions = generate_storage_suggestions(food_name, freshness_status, predicted_life, spoilage_pct)
        
        # H. Clean up temporary uploaded file
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)

        return {
            "success": True,
            "results": {
                "food_name": str(food_name),
                "food_category": str(food_name),
                "freshness_status": str(freshness_status),
                "confidence_score": f"{confidence_score * 100:.2f}%",
                "surface_damage_percentage": f"{spoilage_pct:.1f}%",
                "remaining_shelf_life_days": predicted_life,
                "highlighted_image_base64": f"data:image/jpeg;base64,{highlighted_img}" if highlighted_img else None,
                "bounding_boxes": damage_boxes
            },
            "recommendations": suggestions
        }

    except Exception as e:
        traceback.print_exc()
        if temp_path and os.path.exists(temp_path):
            os.remove(temp_path)
        raise HTTPException(status_code=500, detail=f"Backend Error: {str(e)}")

print("✓ API Routes & XGBoost Columns configured successfully.")

✓ API Routes & XGBoost Columns configured successfully.
